# tiny-llm: a character-level GPT trained on Shakespeare

A small GPT (~10M params) written from scratch in PyTorch: no Hugging Face model classes. Trained on the complete works of Shakespeare (Project Gutenberg #100).

**On Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Training takes roughly 15–25 minutes; the last cell downloads the checkpoint.

In [ ]:
import math
import time
import urllib.request
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## Data

Download the text, keep only the part between Gutenberg's `*** START` / `*** END` markers, and turn curly quotes into straight ones so typed prompts match the training text.

In [ ]:
URL = "https://www.gutenberg.org/cache/epub/100/pg100.txt"
path = Path("data/shakespeare.txt")
if not path.exists():
    path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(URL, path)

raw = path.read_text(encoding="utf-8-sig")
start = raw.index("\n", raw.index("*** START OF")) + 1
end = raw.index("*** END OF")
text = raw[start:end].replace("\r\n", "\n").strip()
text = text.translate(str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"'}))

print(f"{len(text):,} characters")
print(text[10_000:10_500])

## Tokenizer

Character level: every distinct character is one token.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}


def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(chars[i] for i in ids)


assert decode(encode(text[:10_000])) == text[:10_000]
print(vocab_size, "tokens:", repr("".join(chars)))

## Train/val split and batches

First 90% for training, last 10% for validation. A batch is random windows of `block_size` characters; the target is the same window shifted one character to the right.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


def get_batch(split, batch_size, block_size):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


x, y = get_batch("train", 4, 8)
assert x.shape == y.shape == (4, 8)
assert torch.equal(x[:, 1:], y[:, :-1])  # targets are inputs shifted by one

## Model

Pre-norm transformer: token + position embeddings → `n_layer` blocks of (causal multi-head self-attention, MLP) with residuals → final LayerNorm → output head tied to the token embedding.

Attention takes an optional KV cache: the keys/values of earlier positions, so generation only has to run the newest token through the model.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.2


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.register_buffer("mask", torch.tril(torch.ones(cfg.block_size, cfg.block_size, dtype=torch.bool)))

    def forward(self, x, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_head, T, head_size)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        if cache is not None:
            # keys/values of earlier positions go in front of the new ones
            k = torch.cat([cache[0], k], dim=2)
            v = torch.cat([cache[1], v], dim=2)
        S = k.size(2)  # past + current positions
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        # the new queries sit at positions S-T..S-1, each sees keys up to itself
        att = att.masked_fill(~self.mask[S - T:S, :S], float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y)), (k, v)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x, cache=None):
        a, cache = self.attn(self.ln1(x), cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, cache


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layer))
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # tied embeddings
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, caches=None):
        B, T = idx.shape
        past = 0 if caches is None else caches[0][0].size(2)
        assert past + T <= self.cfg.block_size, "sequence longer than block_size"
        pos = torch.arange(past, past + T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        new_caches = []
        for i, block in enumerate(self.blocks):
            x, c = block(x, None if caches is None else caches[i])
            new_caches.append(c)
        logits = self.head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss, new_caches

Check: the mask works, so changing later tokens never changes the outputs at earlier positions.

In [ ]:
tiny = GPTConfig(vocab_size=vocab_size, block_size=64, n_layer=2, n_head=2, n_embd=32, dropout=0.0)
m = GPT(tiny).to(device).eval()
a = torch.randint(vocab_size, (1, 16), device=device)
b = a.clone()
b[0, 10:] = torch.randint(vocab_size, (6,), device=device)
with torch.no_grad():
    la, lb = m(a)[0], m(b)[0]
assert torch.allclose(la[0, :10], lb[0, :10], atol=1e-4)

## Generation

`sample_next` turns the last position's logits into the next token: `temperature=0` is greedy, otherwise temperature scaling plus optional top-k / top-p filtering.

`generate` re-runs the whole window every step. `generate_cached` runs the prompt once and then feeds one token at a time, reusing the KV cache.

In [ ]:
def sample_next(logits, temperature=1.0, top_k=None, top_p=None):
    """logits: (B, vocab) for the last position -> (B, 1) next token ids."""
    if temperature == 0:
        return logits.argmax(dim=-1, keepdim=True)
    logits = logits / temperature
    if top_k is not None:
        kth = torch.topk(logits, min(top_k, logits.size(-1))).values[:, [-1]]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p is not None:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        probs = F.softmax(sorted_logits, dim=-1)
        # drop a token once the tokens before it already cover top_p (the top one always stays)
        sorted_logits[probs.cumsum(dim=-1) - probs >= top_p] = float("-inf")
        logits = torch.full_like(logits, float("-inf")).scatter(-1, sorted_idx, sorted_logits)
    return torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)


@torch.no_grad()
def generate(model, idx, max_new_tokens, **sampling):
    for _ in range(max_new_tokens):
        logits, _, _ = model(idx[:, -model.cfg.block_size:])
        idx = torch.cat([idx, sample_next(logits[:, -1], **sampling)], dim=1)
    return idx


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens, **sampling):
    bs = model.cfg.block_size
    logits, _, caches = model(idx[:, -bs:])
    for _ in range(max_new_tokens):
        nxt = sample_next(logits[:, -1], **sampling)
        idx = torch.cat([idx, nxt], dim=1)
        if caches[0][0].size(2) == bs:
            # ponytail: learned absolute positions can't slide, so a full cache is rebuilt from the last half window
            # (output differs from `generate` past block_size); rotary embeddings would allow a true rolling cache
            logits, _, caches = model(idx[:, -(bs // 2):])
        else:
            logits, _, caches = model(nxt, caches=caches)
    return idx

Checks: cached and uncached greedy decoding agree (within `block_size`), and top-k never picks outside the k best tokens.

In [ ]:
prompt = torch.tensor([encode("ROMEO:")], device=device)
assert torch.equal(generate(m, prompt, 40, temperature=0), generate_cached(m, prompt, 40, temperature=0))

logits = torch.randn(1000, vocab_size)
picked = sample_next(logits, top_k=5)
assert (picked == torch.topk(logits, 5).indices).any(dim=1).all()
print("all checks passed")

## Training

AdamW with linear warmup then cosine decay, gradient clipping, and train/val loss every `eval_interval` steps.

To train the smaller, faster model instead, swap in the commented `cfg` line.

In [ ]:
cfg = GPTConfig(vocab_size=vocab_size)  # ~10.7M params
# cfg = GPTConfig(vocab_size=vocab_size, block_size=128, n_layer=4, n_head=4, n_embd=256)  # ~3M, much faster

batch_size = 64
max_iters = 4000
lr, min_lr = 1e-3, 1e-4
warmup_iters = 200
grad_clip = 1.0
eval_interval = 250
eval_iters = 100


def get_lr(it):
    if it < warmup_iters:
        return lr * (it + 1) / warmup_iters
    progress = (it - warmup_iters) / max(1, max_iters - warmup_iters)
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * progress))


@torch.no_grad()
def estimate_loss(model):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split, batch_size, cfg.block_size)
            losses[k] = model(x, y)[1].item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
model = GPT(cfg).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params")
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99))

log = []
t0 = time.time()
for it in range(max_iters):
    for group in optimizer.param_groups:
        group["lr"] = get_lr(it)
    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss(model)
        log.append((it, losses["train"], losses["val"]))
        print(f"step {it:5d} | train {losses['train']:.4f} | val {losses['val']:.4f} | {time.time() - t0:.0f}s")

    x, y = get_batch("train", batch_size, cfg.block_size)
    _, loss, _ = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

print(f"done in {(time.time() - t0) / 60:.1f} min")

## Samples and KV cache speedup

In [ ]:
model.eval()
prompt = torch.tensor([encode("ROMEO:\n")], device=device)
print(decode(generate_cached(model, prompt, 500, temperature=0.8, top_k=50)[0].tolist()))

In [ ]:
def tokens_per_sec(fn, n=300):
    t = time.time()
    fn(model, prompt, n, temperature=0.8, top_k=50).tolist()  # .tolist() waits for the GPU to finish
    return n / (time.time() - t)


print(f"no cache: {tokens_per_sec(generate):.0f} tok/s")
print(f"kv cache: {tokens_per_sec(generate_cached):.0f} tok/s")

## Save

The checkpoint holds the weights, the config and the vocabulary: everything the inference server needs to rebuild the model. To load it anywhere:

```python
ckpt = torch.load("shakespeare.pt", map_location=device)
model = GPT(GPTConfig(**ckpt["config"])).to(device)
model.load_state_dict(ckpt["model"])
chars = ckpt["chars"]
```

In [ ]:
torch.save({"model": model.state_dict(), "config": asdict(cfg), "chars": chars}, "shakespeare.pt")
with open("losses.csv", "w") as f:
    f.write("step,train,val\n")
    f.writelines(f"{s},{tr:.4f},{va:.4f}\n" for s, tr, va in log)

try:
    from google.colab import files

    files.download("shakespeare.pt")
    files.download("losses.csv")
except ImportError:
    print("saved shakespeare.pt and losses.csv")